## GenAI Disclosure Statement

Generative AI tools were used as learning aids. Analysis and conclusions are the team's own work.

---

# Notebook 03 — Explainability and Proxy-Risk Analysis
### DNSC 6330 Responsible Machine Learning Capstone | GWU

**Purpose:** Use SHAP to explain what the GBM model learned. Identify features that may act
as proxies for race, sex, or ethnicity. Produce counterfactual analysis for denied applicants.

**Inputs:** `data/processed/train.parquet`, `test.parquet`, trained GBM model  
**Outputs:** SHAP plots, `tables/top_features_shap.csv`, `tables/proxy_correlation.csv`,
`tables/counterfactuals.csv`, `figures/shap_*.png`

**Lecture 02 connection:** SHAP global + local explanations, proxy-feature flagging, counterfactuals.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
import shap
import joblib
import glob

from src.leakage import PROXY_RISK_FEATURES
from src.explain import (
    compute_shap_values, top_features_table, proxy_correlation_table,
    generate_counterfactuals
)

SEED = 42
np.random.seed(SEED)

PROC_DIR   = os.path.join(os.getcwd(), "..", "data", "processed")
MODELS_DIR = os.path.join(os.getcwd(), "..", "models")
TABLES_DIR = os.path.join(os.getcwd(), "..", "tables")
FIG_DIR    = os.path.join(os.getcwd(), "..", "figures")


In [ ]:
# Load data and model
train_df = pd.read_parquet(os.path.join(PROC_DIR, "train.parquet"))
test_df  = pd.read_parquet(os.path.join(PROC_DIR, "test.parquet"))

NON_FEATURE_COLS = [
    "y", "action_taken", "state_code",
    "derived_race", "derived_sex", "derived_ethnicity", "race_sex_intersection"
]
feature_cols = [c for c in train_df.columns if c not in NON_FEATURE_COLS]

X_train = train_df[feature_cols]
X_test  = test_df[feature_cols]
y_test  = test_df["y"].values

# Load GBM model
gbm_files = sorted(glob.glob(os.path.join(MODELS_DIR, "gbm_v*.pkl")))
if not gbm_files:
    raise FileNotFoundError("No GBM model found. Run Notebook 02 first.")
gbm_path = gbm_files[-1]
gbm_model = joblib.load(gbm_path)
print(f"Loaded model: {gbm_path}")
print(f"Test set: {len(X_test):,} rows, {len(feature_cols)} features")


## Section 3.1 — Global SHAP (TreeSHAP)

We use a background sample of 2,000 training observations for efficiency.
TreeSHAP produces exact Shapley values for tree-based models.


In [ ]:
# SHAP on a background sample for efficiency
background = shap.sample(X_train, 2000, random_state=SEED)
print("Computing SHAP values on background sample...")
explainer = shap.TreeExplainer(gbm_model, data=background, feature_perturbation="interventional")
shap_values_test = explainer(X_test)
print(f"SHAP values computed. Shape: {shap_values_test.values.shape}")


In [ ]:
# SHAP bar chart — mean |SHAP value|
plt.figure(figsize=(10, 7))
shap.plots.bar(shap_values_test, max_display=20, show=False)
plt.title("Top 20 Features — Mean |SHAP Value| (GBM, Test Set)", fontsize=13)
plt.tight_layout()
bar_path = os.path.join(FIG_DIR, "shap_bar_top20.png")
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {bar_path}")


In [ ]:
# SHAP beeswarm plot
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values_test, max_display=20, show=False)
plt.title("SHAP Beeswarm — Feature Effect Distributions (GBM, Test Set)", fontsize=13)
plt.tight_layout()
beeswarm_path = os.path.join(FIG_DIR, "shap_beeswarm.png")
plt.savefig(beeswarm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {beeswarm_path}")


## Section 3.2 — Top-20 Feature Table with Proxy-Risk Ratings

Every feature in the top 20 is assigned a proxy-risk level (Low / Medium / High) and
a one-line justification. Features rated Medium or High require monitoring.


In [ ]:
# Build top-20 feature table with proxy-risk annotations
top20_df = top_features_table(shap_values_test, X_test, proxy_risk_dict=PROXY_RISK_FEATURES, n=20)

# Flag any High proxy-risk features in bold for the report
high_risk = top20_df[top20_df["proxy_risk_level"] == "High"]
medium_risk = top20_df[top20_df["proxy_risk_level"] == "Medium"]

print(f"Top 20 features — proxy-risk summary:")
print(f"  High risk:   {len(high_risk)} features")
print(f"  Medium risk: {len(medium_risk)} features")
print(f"  Low risk:    {len(top20_df) - len(high_risk) - len(medium_risk)} features")

display(top20_df)
top20_df.to_csv(os.path.join(TABLES_DIR, "top_features_shap.csv"), index=False)
print("\nTop features table saved.")


## Section 3.3 — Proxy Correlation Analysis

We compute Pearson correlation between the top features and label-encoded protected attributes.
Features with |r| > 0.30 vs. any protected attribute are flagged as proxy-risk concerns.


In [ ]:
# Label-encode protected attributes for correlation computation
from sklearn.preprocessing import LabelEncoder

test_meta = test_df[NON_FEATURE_COLS].copy()
for col in ["derived_race", "derived_sex", "derived_ethnicity"]:
    le = LabelEncoder()
    test_meta[f"{col}_enc"] = le.fit_transform(test_meta[col].astype(str))

# Combine features + encoded protected attrs for correlation
corr_df = X_test.copy()
for col in ["derived_race", "derived_sex", "derived_ethnicity"]:
    corr_df[f"{col}_enc"] = test_meta[f"{col}_enc"].values

top_features_list = top20_df["feature"].tolist()

proxy_corr = proxy_correlation_table(
    X=corr_df,
    protected_cols={
        "Race": "derived_race_enc",
        "Sex": "derived_sex_enc",
        "Ethnicity": "derived_ethnicity_enc",
    },
    top_features=top_features_list,
)

print("Proxy correlation table (top 20 features vs. protected attributes):")
display(proxy_corr)
proxy_corr.to_csv(os.path.join(TABLES_DIR, "proxy_correlation.csv"), index=False)

# Flag high-correlation features
for col in ["corr_Race", "corr_Sex", "corr_Ethnicity"]:
    if col in proxy_corr.columns:
        flagged = proxy_corr[proxy_corr[col].abs() > 0.30]
        if len(flagged) > 0:
            print(f"\n⚠ Features with |r| > 0.30 vs. {col.split('_')[1]}:")
            print(flagged[["feature", col]].to_string(index=False))


## Section 3.4 — Local SHAP Explanations

We examine 6 individual cases to understand how the model reasons for specific applicants.
Cases selected to cover: approved majority-group, approved minority-group, denied majority-group,
denied minority-group, and two borderline cases near the decision threshold.


In [ ]:
# Get model predictions on test set
from sklearn.metrics import roc_auc_score
import glob

# Reload the threshold from metrics
metrics_df = pd.read_csv(os.path.join(TABLES_DIR, "metrics_table_final.csv"))
gbm_threshold_row = metrics_df[metrics_df["model"].str.contains("GBM")]
gbm_threshold = float(gbm_threshold_row["threshold"].values[0]) if len(gbm_threshold_row) > 0 else 0.5

test_probs = gbm_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= gbm_threshold).astype(int)

test_full = test_df.copy()
test_full["y_prob"] = test_probs
test_full["y_pred"] = test_preds

# Select representative cases
cases = {}

# Approved White Male (TP — true positive)
tp_white = test_full[(test_full["y"]==1) & (test_full["y_pred"]==1) &
                     (test_full["derived_race"]=="White") & (test_full["derived_sex"]=="Male")]
if len(tp_white) > 0:
    cases["TP_White_Male"] = tp_white.index[0]

# Approved Black applicant (TP)
tp_black = test_full[(test_full["y"]==1) & (test_full["y_pred"]==1) &
                     (test_full["derived_race"]=="Black or African American")]
if len(tp_black) > 0:
    cases["TP_Black"] = tp_black.index[0]

# Correctly denied White applicant (TN)
tn_white = test_full[(test_full["y"]==0) & (test_full["y_pred"]==0) &
                     (test_full["derived_race"]=="White")]
if len(tn_white) > 0:
    cases["TN_White"] = tn_white.index[0]

# Falsely denied Black applicant (FN — model error on minority group)
fn_black = test_full[(test_full["y"]==1) & (test_full["y_pred"]==0) &
                     (test_full["derived_race"]=="Black or African American")]
if len(fn_black) > 0:
    cases["FN_Black_Denied_Incorrectly"] = fn_black.index[0]

# Borderline cases (close to threshold)
borderline = test_full[(test_full["y_prob"].between(gbm_threshold - 0.05, gbm_threshold + 0.05))]
if len(borderline) > 0:
    cases["Borderline_1"] = borderline.index[0]
if len(borderline) > 1:
    cases["Borderline_2"] = borderline.index[1]

print(f"Selected {len(cases)} cases for local explanation:")
for label, idx in cases.items():
    row = test_full.loc[idx]
    print(f"  {label}: idx={idx}, race={row['derived_race']}, sex={row['derived_sex']}, "
          f"y={row['y']}, y_prob={row['y_prob']:.4f}")


In [ ]:
# SHAP waterfall plots for each case
for case_label, case_idx in cases.items():
    # Find position in X_test (by index)
    pos = X_test.index.get_loc(case_idx) if case_idx in X_test.index else None
    if pos is None:
        # Try positional
        pos = list(test_df.index).index(case_idx) if case_idx in test_df.index else 0

    try:
        plt.figure(figsize=(10, 5))
        shap.plots.waterfall(shap_values_test[pos], max_display=12, show=False)
        plt.title(f"SHAP Waterfall — {case_label}\n"
                  f"(y={test_full.loc[case_idx,'y']}, "
                  f"p={test_full.loc[case_idx,'y_prob']:.3f}, "
                  f"race={test_full.loc[case_idx,'derived_race']})", fontsize=10)
        plt.tight_layout()
        save_path = os.path.join(FIG_DIR, f"shap_waterfall_{case_label}.png")
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Could not plot {case_label}: {e}")


## Section 3.5 — Counterfactual Analysis

For denied applicants, we identify the minimal feature changes that would flip the model
prediction from denied to approved. This surfaces the practical actionability of the model's
decisions and flags whether required changes implicate protected-class-correlated features.


In [ ]:
from src.explain import generate_counterfactuals

# Select denied applicants for counterfactual analysis
denied_mask = (test_full["y_pred"] == 0) & (test_full["y"] == 0)  # true denials
X_denied = X_test[denied_mask].reset_index(drop=True)
print(f"Analyzing counterfactuals for {min(20, len(X_denied))} denied applicants...")

# Feature deltas to test — credit-relevant features only (not proxies)
feature_deltas = {
    "income": 20000,             # +$20k stated income
    "loan_amount": -25000,       # -$25k loan amount
    "dti_numeric": -5.0,         # -5 DTI points
    "combined_loan_to_value_ratio": -5.0,  # -5 LTV points
}

# Filter deltas to features that actually exist in X_denied
feature_deltas = {k: v for k, v in feature_deltas.items() if k in X_denied.columns}
print(f"Testing {len(feature_deltas)} feature changes: {list(feature_deltas.keys())}")

cf_df = generate_counterfactuals(
    gbm_model, X_denied,
    feature_deltas=feature_deltas,
    threshold=gbm_threshold,
    n_cases=20,
)

flipped = cf_df[cf_df["decision_flipped"] == True]
print(f"\nCounterfactual results:")
print(f"  Total flips found: {len(flipped)}")
print(f"  Unique cases with a flip: {flipped['case_index'].nunique()}")

# Add proxy-risk annotation to counterfactuals
cf_df["proxy_risk"] = cf_df["feature_changed"].map({
    k: v.get("risk_level", "Low") for k, v in PROXY_RISK_FEATURES.items()
}).fillna("Low")

display(cf_df[cf_df["decision_flipped"] == True][
    ["case_index", "feature_changed", "original_value", "counterfactual_value",
     "original_prob", "counterfactual_prob", "proxy_risk"]
].head(20))

cf_df.to_csv(os.path.join(TABLES_DIR, "counterfactuals.csv"), index=False)
print("\nCounterfactual table saved.")


## Section 3.6 — Explainability Findings

Summarize findings for the audit record.


In [ ]:
print("=" * 60)
print("EXPLAINABILITY FINDINGS — AUDIT SUMMARY")
print("=" * 60)

n_high = len(top20_df[top20_df["proxy_risk_level"] == "High"])
n_med  = len(top20_df[top20_df["proxy_risk_level"] == "Medium"])
n_low  = len(top20_df[top20_df["proxy_risk_level"] == "Low"])

top1 = top20_df.iloc[0]
print(f"\n1. Top driver: {top1['feature']} (mean |SHAP| = {top1['mean_abs_shap']:.4f})")
print(f"   Proxy risk level: {top1['proxy_risk_level']}")

print(f"\n2. Proxy-risk summary (top 20 features):")
print(f"   High proxy risk: {n_high} features")
print(f"   Medium proxy risk: {n_med} features")
print(f"   Low proxy risk:  {n_low} features")

if n_high > 0:
    print(f"\n⚠ HIGH-PROXY-RISK features in top 20:")
    for _, r in top20_df[top20_df["proxy_risk_level"] == "High"].iterrows():
        print(f"   Rank {r['rank']}: {r['feature']} — {r['proxy_justification'][:80]}...")

cf_flips = len(cf_df[cf_df["decision_flipped"] == True])
print(f"\n3. Counterfactual analysis: {cf_flips} decision flips found across")
print(f"   {len(feature_deltas)} tested feature changes for 20 denied applicants.")

print("\n Notebook 03 complete.")
